In [6]:
!pip install ultralytics
from ultralytics import YOLO
import ultralytics

# Sistem uygun mu diye kontrol
ultralytics.checks()

Ultralytics 8.3.203 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6411.5/8062.4 GB disk)


##### Eğitim ayarları:
##### - epochs=200 → maksimum eğitim epoch sayısı
##### - patience=5 → 5 epoch boyunca gelişme olmazsa erken durdurma
##### - batch=8 → GPU uygunsa artırılabilir
##### - optimizer=AdamW → daha stabil öğrenme
##### - lr0=0.01 → başlangıç öğrenme oranı (daha güvenli)
##### - lrf=0.001 → son epoch’a doğru öğrenme oranı
##### - weight_decay=0.0005 → regularization, overfitting’i azaltır
##### - cos_lr=True → Cosine LR Scheduler, daha dengeli öğrenme oranı düşüşü
##### - save_period=10 → her 10 epoch’ta bir checkpoint kaydeder
##### - device=0 → GPU kullan
##### - imgsz=640 → giriş resim boyutu
##### - workers=8 → dataloader iş parçacığı sayısı

In [9]:
import textwrap

yaml_content = textwrap.dedent("""\
    train: /kaggle/input/surugenbocekfinaldataset/augmented_dataset/train/images
    val: /kaggle/input/surugenbocekfinaldataset/augmented_dataset/valid/images
    test: /kaggle/input/surugenbocekfinaldataset/augmented_dataset/test/images

    nc: 22
    names: [
      'Akdeniz Munzevi Orumcegi',
      'Anadolu Sari Akrebi',
      'Kara Akrep',
      'Katil Ari',
      'Yaprak Biti',
      'Kahverengi Kokarca Bocegi',
      'Lahana Tirtili',
      'Patates Bocegi',
      'Misir Kurdu',
      'Misir Yuvarlak Kurdu',
      'Sonbahar Ordu Kurdu',
      'Sirke Sinegi',
      'Kum Yengeci',
      'Uc Benekli Yuzen Yengec',
      'Kirmizi Orumcek',
      'Trips',
      'Mavi Yengec',
      'Kemanci Yengec',
      'Baklagil Kabarcik Bocegi',
      'Camur Yengeci',
      'Pirinc Gal Sinegi',
      'Beyaz Sirtli Bitki Zararlisi'
    ]
""")

with open("/kaggle/working/data.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content)


In [ ]:
!yolo detect train \
    data=/kaggle/working/data.yaml \
    model=yolo11n.pt \
    epochs=200 \
    patience=5 \
    imgsz=640 \
    workers=8 \
    batch=8 \
    device=0 \
    optimizer=AdamW \
    lr0=0.01 \
    lrf=0.001 \
    weight_decay=0.0005 \
    save_period=10 \
    name=SurungenBocek_detection

Ultralytics 8.3.203 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=SurungenBocek_detection, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=5, perspective=0.0, plots=

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/working/runs/detect/SurungenBocek_detection/results.csv")
print("Son epoch:", df['epoch'].max())
print(df.tail())  # Son birkaç epoch'u gösterir

In [ ]:
# Tahmin için gerekli kütüphaneler
import cv2
import imutils
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Yazı tipi
font = cv2.FONT_HERSHEY_SIMPLEX

# Test görseli
img_path = "/kaggle/input/surugenbocekfinaldataset/augmented_dataset/valid/images/108_jpg.rf.4d919027ee8b884f1f06978e23b51461_aug2.jpg"
model_path = "/kaggle/working/runs/detect/SurungenBocek_detection/weights/best.pt"

# Görseli oku ve yeniden boyutlandır
img = cv2.imread(img_path)
img = imutils.resize(img, width=640)

# Modeli yükle ve tahmin yap
model = YOLO(model_path)
results = model(img)[0]

# Eşik değeri
threshold = 0.5

# Tespitleri işle
for result in results.boxes.data.tolist():
    x1, y1, x2, y2, score, class_id = result
    x1, y1, x2, y2, class_id = int(x1), int(y1), int(x2), int(y2), int(class_id)

    if score > threshold:
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        class_name = results.names[class_id]
        score = score * 100
        text = f"{class_name}: %{score:.2f}"
        cv2.putText(img, text, (x1, y1 - 10), font, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

# Görseli BGR'dan RGB'ye çevir ve matplotlib ile göster
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8, 6))
plt.imshow(img_rgb)
plt.title("Tespit Sonucu")
plt.axis("off")
plt.show()

In [ ]:
# Model çıktılarını zip'le
!zip -r SurungenBocek_detection.zip /kaggle/working/runs/detect/SurungenBocek_detection